In [11]:


# hindi_kal_hinglish.py

def detect_tense(sentence):

    sentence = sentence.lower()

    # Past tense indicators (Roman Hindi)
    past_words = ["tha", "thi", "the", "gaya", "gayi", "kiya", "hua"]

    #  Future tense indicators
    future_words = ["jaunga", "jaungi", "jayega", "hoga", "karunga", "aayega","hey"]

    # Check past
    for word in past_words:
        if word in sentence:
            return "Yesterday (Past)"

    # Check future
    for word in future_words:
        if word in sentence:
            return "Tomorrow (Future)"

    return "Ambiguous (Not enough context)"


def main():
    print(" *Hindi 'kal' Ambiguity Resolver (Hinglish Mode)")
    print("Type 'exit' to quit\n")

    while True:
        sentence = input("->Enter sentence (Hinglish): ")

        if sentence.lower() == "exit":
            print("Exiting...")
            break

        if "kal" not in sentence.lower():
            print(" ->Please include the word 'kal'.")
            continue

        result = detect_tense(sentence)
        print(" *Meaning of 'kal':", result)


if __name__ == "__main__":
    main()


 *Hindi 'kal' Ambiguity Resolver (Hinglish Mode)
Type 'exit' to quit

->Enter sentence (Hinglish): kal subha
 *Meaning of 'kal': Ambiguous (Not enough context)
->Enter sentence (Hinglish): kal barish tha
 *Meaning of 'kal': Yesterday (Past)
->Enter sentence (Hinglish): exit
Exiting...


In [12]:
# Transformer for "Kal" Ambiguity Detection (FINAL VERSION)

import torch
import torch.nn as nn
import torch.optim as optim

#  Dataset
data = [
    ("kal me bazar gaya", 0),
    ("kal barish hua", 0),
    ("kal me school gaya tha", 0),
    ("kal me bazar jaunga", 1),
    ("kal meeting hoga", 1),
    ("kal me ghar jaungi", 1),
]

labels_map = {
    0: "Yesterday (Past)",
    1: "Tomorrow (Future)"
}


#  Build Vocabulary

all_words = []
for sentence, _ in data:
    all_words.extend(sentence.split())

vocab = {word: i+1 for i, word in enumerate(set(all_words))}
vocab["<pad>"] = 0

# Encode Function

max_len = 6

def encode(sentence):
    tokens = sentence.lower().split()
    ids = [vocab.get(word, 0) for word in tokens]

    if len(ids) < max_len:
        ids += [0] * (max_len - len(ids))
    else:
        ids = ids[:max_len]

    return ids

X = torch.tensor([encode(s) for s, _ in data])
y = torch.tensor([label for _, label in data])


#  Positional Encoding

class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, embed_dim)

        for pos in range(max_len):
            for i in range(0, embed_dim, 2):
                pe[pos, i] = torch.sin(torch.tensor(pos / (10000 ** ((2*i)/embed_dim))))
                if i+1 < embed_dim:
                    pe[pos, i+1] = torch.cos(torch.tensor(pos / (10000 ** ((2*(i+1))/embed_dim))))

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


#  Transformer Model

class MiniTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, num_heads=2):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_encoding = PositionalEncoding(embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.fc = nn.Linear(embed_dim, 2)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        x = self.transformer(x)

        x = x.mean(dim=1)
        return self.fc(x)


# Train Model

model = MiniTransformer(len(vocab))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

print("Training model... Please wait\n")

for epoch in range(60):
    optimizer.zero_grad()

    outputs = model(X)
    loss = criterion(outputs, y)

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

#  Prediction Function

def predict(sentence):
    model.eval()
    encoded = torch.tensor([encode(sentence)])

    with torch.no_grad():
        output = model(encoded)
        pred = torch.argmax(output, dim=1).item()

    return labels_map[pred]

# User Input

print("\n *Hindi 'kal' Ambiguity Resolver (Transformer Model)")
print("Type 'exit' to quit\n")

while True:
    sentence = input("-> Enter sentence (Hinglish): ")

    if sentence.lower() == "exit":
        print("Exiting...")
        break

    if "kal" not in sentence.lower():
        print(" Please include the word 'kal'.\n")
        continue

    result = predict(sentence)
    print(" Meaning of 'kal':", result, "\n")

Training model... Please wait

Epoch 0, Loss: 0.7267
Epoch 10, Loss: 0.2219
Epoch 20, Loss: 0.0287
Epoch 30, Loss: 0.0042
Epoch 40, Loss: 0.0035
Epoch 50, Loss: 0.0010

 *Hindi 'kal' Ambiguity Resolver (Transformer Model)
Type 'exit' to quit

-> Enter sentence (Hinglish): kal kya hoga
 Meaning of 'kal': Tomorrow (Future) 

-> Enter sentence (Hinglish): kal kya hua
 Meaning of 'kal': Yesterday (Past) 

-> Enter sentence (Hinglish): kal subha
 Meaning of 'kal': Tomorrow (Future) 

-> Enter sentence (Hinglish): kal subha
 Meaning of 'kal': Tomorrow (Future) 

-> Enter sentence (Hinglish): exit
Exiting...
